In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

Setting OPENAI_API_KEY...
Setting GOOGLE_API_KEY...
Setting ANTHROPIC_API_KEY...


# LangChain Common Expression Language (LCEL)

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Create components
prompt = PromptTemplate.from_template("Tell me a joke about {topic}")
llm = ChatOpenAI()
output_parser = StrOutputParser()

# Chain them together using LCEL
chain = prompt | llm | output_parser

# Use the chain
result = chain.invoke({"topic": "programming"})
print(result)


Why do programmers prefer dark mode? Because light attracts bugs!


# More complex expressions

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# chat = ChatGoogleGenerativeAI(model="gemini-pro")
chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# Define a printer function - compact "print and passthrough"
printer = lambda s: (print(s), s)[1]

# First chain generates a story
story_prompt = PromptTemplate.from_template("Write a short story about {topic}")
# story_chain = story_prompt | chat | StrOutputParser() | printer
story_chain = story_prompt | chat | StrOutputParser()

# Second chain analyzes the story
analysis_prompt = PromptTemplate.from_template("Analyze the following story's mood:\n{story}")
analysis_chain = analysis_prompt | chat | StrOutputParser()

output_prompt = PromptTemplate.from_template(
    "Here's the story: \n{story}\n\nHere's the mood: \n{mood}"
)
# Combine chains
story_with_analysis = story_chain | analysis_chain

# Run the combined chain
result = story_with_analysis.invoke({"topic": "a rainy day"})
print(result)


In [ ]:
from langchain_core.runnables import RunnablePassthrough

# using RunnablePassthrough.assign to preserve data - basically we're building a dictionary of keys (dict_keys) as we go
enhanced_chain = RunnablePassthrough.assign(
    story=story_chain, # add 'story' key with generated content
).assign(
    analysis=analysis_chain # add 'analysis' key with analysis of the story
)

# execute the chain
result = enhanced_chain.invoke({"topic": "a rainy day"})
print(result.keys())

# print(result["story"])
print("printing analysis...\n")
print(result["analysis"])

In [ ]:
# for even more control, we can construct dictionaries manually
from operator import itemgetter

manual_chain = (
    RunnablePassthrough() | # pass through input
    {
        "story": story_chain, # add story result
        "topic": itemgetter("topic"), # preserve original topic
    } |
    RunnablePassthrough().assign( # add analysis based on story
        analysis=analysis_chain 
    )
)

result = manual_chain.invoke({"topic": "a rainy day"})
print(result.keys())


In [9]:
# simplified dictionary construction using LCEL
simple_dict_chain_corrected = story_chain | {
    "story": RunnablePassthrough(), # pass the story output as 'story'
    "analysis": analysis_chain
}

# analysis_chain will receive {'story': 'the actual story content'} as expected.
result_corrected = simple_dict_chain_corrected.invoke({"topic": "a rainy day"})
print(result_corrected.keys())


The sky was a bruised plum color, swollen with the promise of rain. And then, it arrived. Not a gentle patter, but a furious downpour that hammered against the windowpanes like a frantic drummer.

Elara sighed, abandoning her half-finished painting. The vibrant hues she’d been painstakingly blending suddenly felt dull, muted by the oppressive grey outside. The rain had stolen the light, and with it, her inspiration.

She poured herself a cup of Earl Grey, the steam a temporary comfort against the damp chill that had seeped into the old cottage. The rhythmic drumming of the rain was a lullaby, a melancholic tune that echoed the loneliness she felt.

Outside, the world was transformed. The garden, usually a riot of color, was now a blur of greens and browns, the flowers bowed under the weight of the relentless rain. The path to the gate was a glistening stream, reflecting the somber sky.

Suddenly, a flash of movement caught her eye. A small, bedraggled figure huddled beneath the ancient